In [5]:
import json

# read output file from the solver
# i gotta go up one directory to get to the outputs folder
output_file = "../outputs/sydney_routes_20260923_013228.json"

with open(output_file, "r") as f:
    result = json.load(f)

routes = {}

for route in result["routes"]:
    # i need to create a list of tuples for the locations
    # stop doesnt have lat or lon, but it has a name so i will use that to get the lat and lon from the stops in the result
    # optimizer is also one directory up, so i will import the geocode_address function from optimizer.geocoder
    import sys
    from pathlib import Path

    sys.path.insert(0, str(Path.cwd().parent))
    from optimizer.geocoder import geocode_address
    #geocode_address returns dict with lat and lon, so i will convert it to a tuple
    locations = [geocode_address(stop["name"]) for stop in route["stops"]]
    routes[route["vehicle_id"]] = [(location["latitude"], location["longitude"]) for location in locations]

    import folium
    # create a folium map centered on the first location
    m = folium.Map(
        location=routes[route["vehicle_id"]][0],
        zoom_start=13
        , width="100%", height="100%"
    )

    # create a color dictionary for the vehicles
    colors = {}
    for i, vehicle in enumerate(routes.keys()):
        colors[vehicle] = f"#{i*50:02x}{(255-i*50):02x}00"

    # Depot marker
    folium.Marker(
        [routes[route["vehicle_id"]][0][0], routes[route["vehicle_id"]][0][1]],
        popup="Depot",
        tooltip="Depot",
        icon=folium.Icon(color="black", icon="home")
    ).add_to(m)

    # Add routes and customer markers
    for vehicle, route in routes.items():

        # Route
        folium.PolyLine(
            route,
            color=colors[vehicle],
            weight=5,
            opacity=0.8,
            tooltip=vehicle
        ).add_to(m)

        # Customer markers
        for i, location in enumerate(route[1:-1], start=1):
            folium.Marker(
                location,
                popup=f"{vehicle} - Stop {i}",
                tooltip=f"Stop {i}",
                icon=folium.Icon(color=colors[vehicle])
            ).add_to(m)

fig = folium.Figure(width=800, height=400)
fig.add_child(m)
display(fig)

-> Geocoding: '43 Bourke Rd, Alexandria NSW 2015, Sydney, NSW, Australia'...
-> Geocoding: '43 Bourke Rd, Alexandria NSW 2015, Sydney, NSW, Australia'...
-> Geocoding: 'RPA Hospital (Camperdown), Sydney, NSW, Australia'...
-> Geocoding: '43 Bourke Rd, Alexandria NSW 2015, Sydney, NSW, Australia'...
-> Geocoding: '345 Victoria Ave, Chatswood NSW 2067, Sydney, NSW, Australia'...
-> Geocoding: 'Royal North Shore Hospital (St Leonards), Sydney, NSW, Australia'...
-> Geocoding: '500 Oxford St, Bondi Junction NSW 2022, Sydney, NSW, Australia'...
-> Geocoding: 'St Vincent's Hospital (Darlinghurst), Sydney, NSW, Australia'...


/var/folders/mg/6h85ldxn33b9v5n8j1m3fxcw0000gn/T/ipykernel_90423/4047595808.py:64: UserWarning: color argument of Icon should be one of: {'darkgreen', 'beige', 'green', 'cadetblue', 'red', 'darkred', 'white', 'blue', 'black', 'pink', 'gray', 'lightblue', 'lightred', 'lightgreen', 'darkpurple', 'orange', 'purple', 'darkblue', 'lightgray'}.
  icon=folium.Icon(color=colors[vehicle])
